# SVAMITVA Rooftop Classification: Partitioned Training Node
**Author:** Ram  
**Objective:** Train EfficientNet B0 on rooftop crops using the Incremental/Partitioned Strategy.

---

## Step 1: Environment Setup
Clone the repo and install dependencies. This must be done every time the session restarts.

In [ ]:
import os
if not os.path.exists('AI-Powered-Feature-Extraction-System'):
    !git clone https://github.com/print-ramcharan/AI-Powered-Feature-Extraction-System.git
    %cd AI-Powered-Feature-Extraction-System
else:
    %cd AI-Powered-Feature-Extraction-System

!pip install -r requirements.txt

## Step 2: Download Partitioned Data
Download specific volumes from Kaggle. Each volume is a separate dataset (e.g., `chhattisgarh-vol1`, `chhattisgarh-vol2`).

In [ ]:
# Set which volume to train (1-24)
VOL_NUM = 1 

# Setup Kaggle API (Ensure kaggle.json is uploaded to root)
!mkdir -p /root/.kaggle
if os.path.exists('/content/kaggle.json'):
    !cp /content/kaggle.json /root/.kaggle/ 
!chmod 600 /root/.kaggle/kaggle.json

# Download the specific volume dataset
!kaggle datasets download -d pramcharanteja/chhattisgarh-vol{VOL_NUM}
!unzip -q chhattisgarh-vol{VOL_NUM}.zip -d data/raw/partitioned/vol{VOL_NUM}

## Step 3: Mandatory Subset Verification
Test the training loop on a tiny subset (50 images) before committing to full scale.

In [ ]:
import sys
sys.path.append('src')
from train_rooftype import train_rooftype_model

DATA_DIR = f"data/raw/partitioned/vol{VOL_NUM}/"

# Launch tiny verification run
print("Initiating mandatory subset verification...")
train_rooftype_model(DATA_DIR, epochs=1, batch_size=4, save_path="models/subset_test.pt")
print("Subset verify successful. Safe to proceed to full training.")

## Step 4: Full Partitioned Training
Train on the full volume and save the weights. Set `CHECKPOINT_PATH` if continuing from Volume X-1.

In [ ]:
# Set checkpoint if continuing from previous volume (e.g., vol1 output)
CHECKPOINT_PATH = None # Change to "models/rooftype_vol{VOL_NUM-1}.pt" if applicable
SAVE_PATH = f"models/rooftype_vol{VOL_NUM}.pt"

train_rooftype_model(
    data_dir=DATA_DIR, 
    epochs=15, 
    batch_size=32, 
    checkpoint_path=CHECKPOINT_PATH, 
    save_path=SAVE_PATH
)